<a href="https://colab.research.google.com/github/jiyeonlee-2930/HandsOn-LLM/blob/main/BERT_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 이 셀은 BERT 모델을 불러오고 학습하는 데 필요한 라이브러리를 설치합니다.
- transformers: Hugging Face의 사전학습 언어모델과 토크나이저를 사용하는 라이브러리입니다.
- torch: 텐서 연산, 자동 미분, 딥러닝 모델 학습을 담당하는 PyTorch 라이브러리입니다.
- !pip: Colab 노트북에서 운영체제 명령을 실행합니다.
- -q: 설치 로그를 간략하게 출력합니다.
실습 순서: 이 셀을 먼저 실행한 다음 아래 셀을 차례대로 실행합니다.

In [ ]:
!pip -q install transformers torch

# 다국어 BERT 모델과 토크나이저 불러오기
이 셀은 한국어를 포함한 여러 언어를 처리할 수 있는 사전학습 BERT를 불러와 3개 주제 분류 모델로 준비합니다.
- import torch: PyTorch 사용 준비
- AutoTokenizer: 문장을 BERT가 처리할 수 있는 토큰 ID로 변환
- AutoModelForSequenceClassification: BERT 인코더 위에 문장 분류용 출력층을 붙인 모델
- model_name: 사용할 사전학습 모델의 식별자
- from_pretrained(model_name): 사전학습된 토크나이저와 모델 가중치를 다운로드하여 불러옴
- num_labels=3: 분류할 범주가 **도서관(0), 장학금(1), 수강신청(2)**의 세 가지임을 지정
출력되는 구조 정보는 다음을 의미합니다.

#주의: 분류 출력층은 이 실습의 3개 범주에 맞춰 새로 초기화될 수 있으므로, 아래 미세튜닝 과정이 필요합니다. 사전학습 BERT를 불러오는 것만으로 세 범주를 정확하게 분류할 수 있는 것은 아닙니다.

In [ ]:
##버트 불러오기
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

model_name = "google-bert/bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

print("Encoder 층 수:", model.config.num_hidden_layers)
print("Attention Head 수:", model.config.num_attention_heads)
print("은닉 차원:", model.config.hidden_size)

In [ ]:
##학습 데이터 준비
texts = [
    "도서관은 몇 시에 열어요?",
    "도서관 운영 시간을 알려주세요",
    "도서관은 주말에도 이용할 수 있나요?",
    "도서관 열람실은 언제 닫나요?",

    "장학금 신청 기간은 언제인가요?",
    "장학금을 받으려면 어떻게 해야 하나요?",
    "장학금 지급일을 알려주세요",
    "성적 장학금 신청 방법이 궁금합니다",

    "수강신청은 어떻게 하나요?",
    "수강신청 변경 기간은 언제인가요?",
    "이번 학기 수강신청 일정을 알려주세요",
    "수강신청 취소 방법이 궁금합니다"
]

# 0: 도서관, 1: 장학금, 2: 수강신청
labels = [0]*4 + [1]*4 + [2]*4

inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

labels = torch.tensor(labels)
print(inputs["input_ids"].shape)

# 주의: 분류 출력층은 이 실습의 3개 범주에 맞춰 새로 초기화될 수 있으므로, 아래 미세튜닝 과정이 필요합니다. 사전학습 BERT를 불러오는 것만으로 세 범주를 정확하게 분류할 수 있는 것은 아닙니다.

In [ ]:
##미세튜닝 실행
from torch.optim import AdamW

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}
labels = labels.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

model.train()

for epoch in range(10):
    optimizer.zero_grad()

    outputs = model(
        **inputs,
        labels=labels
    )

    loss = outputs.loss

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

In [ ]:
# 이미 미세튜닝 루프(of2E5OvGKsB0 셀) 내에서 역전파가 실행되었으므로
# 이 셀을 중복 실행하면 그래프가 해제되어 에러가 발생합니다.
# loss = outputs.loss
# loss.backward()
# optimizer.step()

In [ ]:
model.eval()

test_text = "장학금은 언제 받을 수 있나요?"

test_input = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=64
)

test_input = {
    k: v.to(device)
    for k, v in test_input.items()
}

with torch.no_grad():
    outputs = model(**test_input)
    pred = torch.argmax(outputs.logits, dim=1).item()

classes = ["도서관", "장학금", "수강신청"]

print("질문:", test_text)
print("예측:", classes[pred])